# Step 7: Random Forest 3-Variant Benchmark Ablation Study
**MSc Data Science Thesis — University of Wolverhampton**

###  Objective & Methodological Design
Train and evaluate **3 progressive Random Forest model variants** on the 80/20 data split ($N_{\text{train}} = 88,591$, $N_{\text{test}} = 22,148$) established in Step 6.5:

1. **Variant 1 (Cost-Unaware RF)**: Standard Random Forest without sample cost weighting or threshold tuning ($t = 0.50$).
2. **Variant 2 (Cost-Aware RF)**: Random Forest trained with Elkan sample cost weights ($w_i = \text{Cost}(FN_i) / \text{Mean}$) and default threshold ($t = 0.50$).
3. **Variant 3 (Cost-Aware RF + OOF Threshold)**: Random Forest trained with sample cost weights AND 5-Fold Out-of-Fold (OOF) cross-validation decision threshold tuning ($t_{\text{opt}} = 0.26$).

### Benchmark Purpose
This 3-variant ablation study isolates the financial loss reduction delivered by sample weighting alone vs sample weighting combined with OOF threshold tuning prior to Step 8 XGBoost evaluation.

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings('ignore')

print('======================================================================')
print('STEP 7: RANDOM FOREST 3-VARIANT ABLATION EXPERIMENT')
print('======================================================================')

# Load dataset
df = pd.read_csv('data_with_cost_matrix.csv')
print(f'Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
if 'product_category_name_english' in df.columns:
    df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
    feature_cols.append('category_encoded')

feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['is_returned']
w = df['sample_cost_weight'] if 'sample_cost_weight' in df.columns else None

# Stratified 80/20 train/test split (Matching Method 2 from Step 6.5)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.20, random_state=42, shuffle=False
)
test_indices = X_test.index

def compute_financial_loss(y_true, y_pred, df_full, idx_subset):
    fn_mask = (y_true == 1) & (y_pred == 0)
    fp_mask = (y_true == 0) & (y_pred == 1)
    fn_loss = df_full.loc[idx_subset[fn_mask], 'cost_FN'].sum()
    fp_loss = df_full.loc[idx_subset[fp_mask], 'cost_FP'].sum()
    return fn_loss + fp_loss, fn_mask.sum(), fp_mask.sum()

results = []

# --- Variant 1: Cost-Unaware Random Forest (Default t=0.50) ---
print('\n--- Training Variant 1: Cost-Unaware Random Forest (Default t=0.50) ---')
rf1 = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf1.fit(X_train, y_train)
y_pred1 = rf1.predict(X_test)
y_prob1 = rf1.predict_proba(X_test)[:, 1]
loss1, fn1, fp1 = compute_financial_loss(y_test, y_pred1, df, test_indices)
results.append({
    'Variant': 'Variant 1: Cost-Unaware RF',
    'Sample Weights?': 'No', 'Threshold Strategy': 'Default (t=0.50)', 'Threshold (t)': 0.50,
    'Accuracy': f'{accuracy_score(y_test, y_pred1)*100:.2f}%',
    'Precision': f'{precision_score(y_test, y_pred1, zero_division=0)*100:.2f}%',
    'Recall': f'{recall_score(y_test, y_pred1)*100:.2f}%',
    'F1-Score': f'{f1_score(y_test, y_pred1, zero_division=0)*100:.2f}%',
    'AUC-ROC': f'{roc_auc_score(y_test, y_prob1)*100:.2f}%',
    'Total Loss (R$)': f'R$ {loss1:,.2f}', 'Loss Per Order': f'R$ {loss1/len(y_test):.4f}'
})

# --- Variant 2: Cost-Aware Random Forest (Default t=0.50) ---
print('\n--- Training Variant 2: Cost-Aware Random Forest (Default t=0.50) ---')
rf2 = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf2.fit(X_train, y_train, sample_weight=w_train)
y_pred2 = rf2.predict(X_test)
y_prob2 = rf2.predict_proba(X_test)[:, 1]
loss2, fn2, fp2 = compute_financial_loss(y_test, y_pred2, df, test_indices)
results.append({
    'Variant': 'Variant 2: Cost-Aware RF',
    'Sample Weights?': 'Yes', 'Threshold Strategy': 'Default (t=0.50)', 'Threshold (t)': 0.50,
    'Accuracy': f'{accuracy_score(y_test, y_pred2)*100:.2f}%',
    'Precision': f'{precision_score(y_test, y_pred2, zero_division=0)*100:.2f}%',
    'Recall': f'{recall_score(y_test, y_pred2)*100:.2f}%',
    'F1-Score': f'{f1_score(y_test, y_pred2, zero_division=0)*100:.2f}%',
    'AUC-ROC': f'{roc_auc_score(y_test, y_prob2)*100:.2f}%',
    'Total Loss (R$)': f'R$ {loss2:,.2f}', 'Loss Per Order': f'R$ {loss2/len(y_test):.4f}'
})

# --- Variant 3: Cost-Aware Random Forest (5-Fold OOF CV Threshold) ---
print('\n--- Training Variant 3: Cost-Aware Random Forest (5-Fold OOF CV Threshold) ---')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs_rf = np.zeros(len(X_train))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    w_tr = w_train.iloc[tr_idx] if w_train is not None else None
    m_cv = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    if w_tr is not None:
        m_cv.fit(X_tr, y_tr, sample_weight=w_tr)
    else:
        m_cv.fit(X_tr, y_tr)
    oof_probs_rf[val_idx] = m_cv.predict_proba(X_va)[:, 1]

train_indices = X_train.index
best_t_rf = 0.50
min_oof_loss = float('inf')
for t in np.arange(0.05, 0.95, 0.01):
    y_pred_oof = (oof_probs_rf >= t).astype(int)
    fn_mask = (y_train == 1) & (y_pred_oof == 0)
    fp_mask = (y_train == 0) & (y_pred_oof == 1)
    loss = df.loc[train_indices[fn_mask], 'cost_FN'].sum() + df.loc[train_indices[fp_mask], 'cost_FP'].sum()
    if loss < min_oof_loss:
        min_oof_loss = loss
        best_t_rf = t

print(f'Optimal OOF Threshold for RF Variant 3: {best_t_rf:.2f}')
y_pred3 = (y_prob2 >= best_t_rf).astype(int)
loss3, fn3, fp3 = compute_financial_loss(y_test, y_pred3, df, test_indices)
results.append({
    'Variant': 'Variant 3: Cost-Aware RF + OOF Threshold',
    'Sample Weights?': 'Yes', 'Threshold Strategy': f'OOF Tuned (t={best_t_rf:.2f})', 'Threshold (t)': round(best_t_rf, 2),
    'Accuracy': f'{accuracy_score(y_test, y_pred3)*100:.2f}%',
    'Precision': f'{precision_score(y_test, y_pred3, zero_division=0)*100:.2f}%',
    'Recall': f'{recall_score(y_test, y_pred3)*100:.2f}%',
    'F1-Score': f'{f1_score(y_test, y_pred3, zero_division=0)*100:.2f}%',
    'AUC-ROC': f'{roc_auc_score(y_test, y_prob2)*100:.2f}%',
    'Total Loss (R$)': f'R$ {loss3:,.2f}', 'Loss Per Order': f'R$ {loss3/len(y_test):.4f}'
})

res_df = pd.DataFrame(results)
print('\n======================================================================')
print('STEP 7: RANDOM FOREST 3-VARIANT RESULTS SUMMARY')
print('======================================================================')
print(res_df.to_string(index=False))
res_df.to_csv('baseline_results.csv', index=False)
print("\n✅ Saved 3-variant Random Forest benchmark results to 'baseline_results.csv'!")


STEP 7: RANDOM FOREST 3-VARIANT ABLATION EXPERIMENT
Loaded dataset: 110,739 rows x 59 columns

--- Training Variant 1: Cost-Unaware Random Forest (Default t=0.50) ---

--- Training Variant 2: Cost-Aware Random Forest (Default t=0.50) ---

--- Training Variant 3: Cost-Aware Random Forest (5-Fold OOF CV Threshold) ---
Optimal OOF Threshold for RF Variant 3: 0.31

STEP 7: RANDOM FOREST 3-VARIANT RESULTS SUMMARY
                                 Variant Sample Weights? Threshold Strategy  Threshold (t) Accuracy Precision Recall F1-Score AUC-ROC Total Loss (R$) Loss Per Order
              Variant 1: Cost-Unaware RF              No   Default (t=0.50)           0.50   90.96%    80.69% 34.49%   48.32%  76.50%   R$ 103,299.86      R$ 4.6641
                Variant 2: Cost-Aware RF             Yes   Default (t=0.50)           0.50   90.91%    79.78% 34.60%   48.27%  75.63%   R$ 102,796.21      R$ 4.6413
Variant 3: Cost-Aware RF + OOF Threshold             Yes OOF Tuned (t=0.31)           0.31   